In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, accuracy_score,
    precision_score, recall_score, ConfusionMatrixDisplay
)


In [ ]:
df = pd.read_csv("diabetic_data.csv")

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values represented by '?':")
print((df == "?").sum().sort_values(ascending=False).head(15))

print("\nOriginal target distribution:")
print(df["readmitted"].value_counts())


In [ ]:
df["readmission_30"] = (df["readmitted"] == "<30").astype(int)

print(df["readmission_30"].value_counts())
print("\nPercentage distribution:")
print((df["readmission_30"].value_counts(normalize=True) * 100).round(2))


In [ ]:
features = [
    "race", "gender", "age",
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
    "diag_1", "diag_2", "diag_3",
    "max_glu_serum", "A1Cresult", "change", "diabetesMed"
]

X = df[features].copy()
y = df["readmission_30"]

X = X.replace("?", np.nan)

print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
numeric_features = [
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses"
]

categorical_features = [
    "race", "gender", "age", "diag_1", "diag_2", "diag_3",
    "max_glu_serum", "A1Cresult", "change", "diabetesMed"
]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        penalty="l2",
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model training completed.")


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("First 10 predicted classes:", y_pred[:10])
print("First 10 readmission probabilities:", np.round(y_prob[:10], 3))


In [ ]:
roc_auc = roc_auc_score(y_test, y_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

print(f"ROC-AUC Score: {roc_auc:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - 30-Day Hospital Readmission")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Readmitted", "Readmitted <30 days"]
).plot()

plt.title("Confusion Matrix")
plt.show()


In [ ]:
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    y_threshold = (y_prob >= threshold).astype(int)

    tn_t, fp_t, fn_t, tp_t = confusion_matrix(
        y_test, y_threshold
    ).ravel()

    recall_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) else 0
    precision_t = tp_t / (tp_t + fp_t) if (tp_t + fp_t) else 0

    print(
        f"Threshold={threshold:.2f} | "
        f"TP={tp_t}, FP={fp_t}, FN={fn_t}, TN={tn_t} | "
        f"Precision={precision_t:.3f}, Recall={recall_t:.3f}"
    )
